# 02 · Corpus auxiliar: BRIGHTER en español
**Fase 2 de la propuesta (Sección 4.3.4).** No necesita GPU; solo internet para descargar el corpus.

Carga las particiones oficiales del subconjunto en español (configuración `esp`),
aplica el mismo preprocesamiento que a las letras y verifica los tamaños
1996 / 184 / 1695.

In [1]:
# Preparar rutas: la carpeta implementacion/ de tu computadora
import sys, os
from pathlib import Path

BASE = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

CODIGO = BASE / 'codigo'
DATOS = BASE / 'datos'
RESULTADOS = BASE / 'resultados'
DATOS.mkdir(exist_ok=True); RESULTADOS.mkdir(exist_ok=True)
assert (CODIGO / 'datos.py').exists(), f'No encuentro {CODIGO}/datos.py. Abre este notebook desde la carpeta implementacion/notebooks'
sys.path.insert(0, str(CODIGO))

import torch
print('Carpeta del proyecto:', BASE)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no hay; se usará la CPU (más lento)')

Carpeta del proyecto: C:\Users\Usuario\IAT\TESIS
GPU: NVIDIA GeForce RTX 3050


In [2]:
import pandas as pd
from datos import cargar_brighter, resumen_brighter, EMOCIONES

brighter = cargar_brighter(idioma='esp')
resumen_brighter(brighter)       # esperado: train 1996, validacion 184, test 1695

C:\Users\Usuario\anaconda3\envs\TESIS\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


dev: 184 filas duplicadas eliminadas (368 -> 184); textos repetidos con etiquetas distintas: 0
test: 1695 filas duplicadas eliminadas (3390 -> 1695); textos repetidos con etiquetas distintas: 0


,instancias,anger,disgust,fear,joy,sadness,surprise
train,1996,492,654,317,642,309,421
validacion,184,34,65,32,54,31,40
test,1695,403,556,200,541,246,394


In [3]:
for nombre, df in brighter.items():
    df.to_csv(f'{DATOS}/brighter_{nombre}.csv', index=False)
brighter['train'].head()

,id,texto,anger,disgust,fear,joy,sadness,surprise
0,esp_train_track_a_00001,"2018 y lo sigo escuchando like si tú también ,...",0,0,0,1,0,0
1,esp_train_track_a_00002,no les aguas. caso a los pen...sativos de los ...,0,1,0,0,0,0
2,esp_train_track_a_00003,yo ya estoy.cagandome de risa pero jajajaaja e...,0,0,0,1,0,0
3,esp_train_track_a_00004,donde le pongo donde son tontos y asquerosos y...,1,1,0,0,0,0
4,esp_train_track_a_00005,"el ritmo está bien, la letra es un asco y quié...",0,1,0,0,0,0


In [4]:
# Cuántas emociones tiene cada comentario (no hay instancias neutras: el mínimo debería ser 1)
brighter['train'][EMOCIONES].sum(axis=1).value_counts().sort_index()

1    1218
2     718
3      59
4       1
Name: count, dtype: int64

In [5]:
for nombre, df in brighter.items():
    print(nombre, len(df),
          '| ids repetidos:', df['id'].duplicated().sum(),
          '| textos repetidos:', df['texto'].duplicated().sum())
print(brighter['test'].head(4))

train 1996 | ids repetidos: 0 | textos repetidos: 0
validacion 184 | ids repetidos: 0 | textos repetidos: 0
test 1695 | ids repetidos: 0 | textos repetidos: 0
                       id                                              texto  \
0  esp_test_track_a_00001   pues 50k dolares no está tan mal. una coperacha.   
1  esp_test_track_a_00002               wao... me da mas horror cocomelon xd   
2  esp_test_track_a_00003                             ya vienes a criticarme   
3  esp_test_track_a_00004  huy que pavor mejor voy a seguir dormiendo bue...   

   anger  disgust  fear  joy  sadness  surprise  
0      0        0     0    1        0         0  
1      0        0     1    0        0         1  
2      0        1     0    0        0         0  
3      0        0     1    0        0         0  
